#### L5: Automate Event Planning

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai crewai_tools langchain_community
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

## CrewAI Tools

In [2]:
import os
from crewai import Agent, Crew, Task
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [3]:
# Agent 1: Venue Coordinator
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    )
)

In [4]:
 # Agent 2: Logistics Manager
logistics_manager = Agent(
    role='Logistics Manager',
    goal=(
        "Manage all logistics for the event "
        "including catering and equipmen"
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    )
)

In [5]:
# Agent 3: Marketing and Communications Agent
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    )
)

## Creating Venue Pydantic Object

- Create a class `VenueDetails` using [Pydantic BaseModel](https://knowledge.pydantic.dev/latest/api/base_model/).
- Agents will populate this object with information about different venues by creating different instances of it.

In [6]:
from pydantic import BaseModel
# Define a Pydantic model for venue details 
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

## Creating Tasks
- By using `output_json`, you can specify the structure of the output you want.
- By using `output_file`, you can get your output in a file.
- By setting `human_input=True`, the task will ask for human feedback (whether you like the results or not) before finalising it.

- By setting `async_execution=True`, it means the task can run in parallel with the tasks which come after it.

In [7]:
logistics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    human_input=True,
    async_execution=True,
    agent=logistics_manager
)

In [8]:
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    async_execution=True,
    output_file="outputs/L010/marketing_report_{run_id}.md",  # Outputs the report as a text file
    agent=marketing_communications_agent
)

In [9]:
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    human_input=True,
    output_json=VenueDetails,
    output_file="outputs/L010/venue_details_{run_id}.json",  
      # Outputs the venue details as a JSON file
    agent=venue_coordinator,
    context=[logistics_task, marketing_task],
)

## Creating the Crew

**Note**: Since you set `async_execution=True` for `logistics_task` and `marketing_task` tasks, now the order for them does not matter in the `tasks` list.

In [10]:
# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[
        logistics_manager, 
        marketing_communications_agent,
        venue_coordinator
    ],
    tasks=[
        logistics_task, 
        marketing_task,
        venue_task
    ],
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [11]:
from datetime import datetime

#
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "San Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall",
    'run_id':run_id
}

**Note 1**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

**Note 2**: 
- Since you set `human_input=True` for some tasks, the execution will ask for your input before it finishes running.
- When it asks for feedback, use your mouse pointer to first click in the text box before typing anything.

In [12]:
result = event_management_crew.kickoff(inputs=event_details)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d87d7357-d974-46d6-8231-8e97d638062f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Coordinate catering and equipment for an event with 500 participants on 2024-09-15.                      │
│  ID: 59b990fe-b957-4faa-b6d7-23c7f8625de0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Logistics Manager                                                                                       │
│                                                                                                                 │
│  Task: Coordinate catering and equipment for an event with 500 participants on 2024-09-15.                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing and Communications Agent                                                                      │
│                                                                                                                 │
│  Task: Promote the Tech Innovation Conference aiming to engage at least500 potential attendees.                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  name='The Midway' address='900 Marin St. San Francisco, CA 94124' capacity=500 booking_status='Available (but  │
│  requires custom quote and booking process)'                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── 💬 Human Feedback Required ───────────────────────────────────────────╮
│                                                                                                                 │
│  Provide feedback on the Final Result above.                                                                    │
│                                                                                                                 │
│  • If you are happy with the result, simply hit Enter without typing anything.                                  │
│  • Otherwise, provide specific improvement requests.                                                            │
│  • You can provide multiple rounds of feedback until satisfied.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Find a venue in San Francisco that meets criteria for Tech Innovation Conference.                        │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d87d7357-d974-46d6-8231-8e97d638062f                                                                       │
│  Final Output: {"name":"The Midway","address":"900 Marin St. San Francisco, CA                                  │
│  94124","capacity":500,"booking_status":"Available (but requires custom quote and booking process)"}            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 812a495b-b376-4fa0-a9aa-81ebc8e86ff2                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/812a495b-b376-4fa │
│ 0-a9aa-81ebc8e86ff2?access_code=TRACE-2c2e24e279                             │
│ 🔑 Access Code: TRACE-2c2e24e279                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


- Display the generated `venue_details.json` file.

In [ ]:
import json
from pprint import pprint

with open(f"outputs/L010/venue_details_{run_id}.json") as f:
   data = json.load(f)

pprint(data)

{'address': '1 Conoco St, San Francisco, CA 94105',
 'booking_status': 'Available upon inquiry based on dates',
 'capacity': 500,
 'name': 'San Francisco Convention Center'}


- Display the generated `marketing_report.md` file.

**Note**: After `kickoff` execution has successfully ran, wait an extra 45 seconds for the `marketing_report.md` file to be generated. If you try to run the code below before the file has been generated, your output would look like:

```
marketing_report.md
```

If you see this output, wait some more and than try again.

In [16]:
from IPython.display import Markdown
Markdown(f"{os.environ["WORK_DIR"]}/outputs/L010/marketing_report_{run_id}.md")

# 🚀 Tech Innovation Conference: Comprehensive Marketing & Engagement Strategy Report

**Prepared For:** Stakeholders / Event Organizers
**Authored By:** Marketing & Communications Agent
**Date:** October 26, 2023
**Goal:** Achieve minimum engagement of 500 qualified attendees for the Tech Innovation Conference.

---

## ✨ Executive Summary & Core Pillars

Our strategy is built on creating intense scarcity, high perceived value, and professional accountability. We will move beyond standard promotion by positioning the conference not merely as an event, but as **the indispensable epicenter of future industry knowledge and crucial high-level networking.**

**Core Pillars:**
1.  **Thought Leadership:** Leveraging star speakers and breakthrough content.
2.  **Community Building:** Creating pre-event buzz and peer-to-peer engagement.
3.  **Targeted Conversion:** Using data and niche marketing to reach decision-makers.

---

## 🎯 Phase 1: Audience Definition & Messaging (The "Who" and "Why")

Before marketing, we solidify who we are talking to.

| Persona Segment | Pain Points/Needs | Key Marketing Hooks |
| :--- | :--- | :--- |
| **The Executive (Decision Maker)** | Needs actionable ROI; fears falling behind industry curves. | "Future-Proof Your Strategy": Focus on economic impact, investment trends, and C-suite insights. |
| **The Practitioner (Engineer/Developer)** | Needs practical, cutting-edge tools; bored by theory. | "Hands-On Deep Dives": Focus on specific tech stacks, real-world case studies, and workshop attendance. |
| **The Investor/Startup Founder** | Needs validation; seeks high-potential networking connections. | "Connect & Capitalize": Focus on networking algorithms, pitch hours, and connecting with venture capital. |

**Primary Messaging Slogan:** *"Innovate. Connect. Lead. Your Blueprint for Tomorrow, Today."*

---

## 🌐 Phase 2: Integrated Marketing Activities & Channels

We will deploy a three-pronged campaign focusing on PR, Digital, and Experiential marketing.

### 1. Content Marketing (Attracting & Educating)
*   **Speaker Spotlights:** Weekly deep-dive interviews (video and blog) with confirmed speakers. These position the *conference* as the destination where the conversation culminates.
*   **Teaser Webinars:** Host two free, bite-sized webinars themed around "Challenges in [Current Tech Sector]." Use these webinars to gather emails and qualify leads (`Lead Magnet`).
*   **White Paper Distribution:** Develop and promote a proprietary "State of Tech 2024 Index," requiring email opt-in for download.

### 2. Digital Marketing & Social Media (Amplifying & Engaging)
*   **LinkedIn Domination:** Primary focus. Run targeted ad campaigns (LinkedIn Ads) specifically targeting job titles (e.g., "CTO," "Director of Engineering") within relevant industries. Promote the value of *networking* exclusivity.
*   **Hashtag Campaign:** Launch a branded, interactive challenge: **#FutureTechChallenge**. Attendees submit their best predictions for the next big tech breakthrough; winners win VIP pass upgrades.
*   **Influencer Outreach:** Partner with 3-5 established tech journalists and niche thought leaders. Offer them complimentary access/speaking slots in exchange for organic promotion to their professional networks.

### 3. Public Relations & Partnerships (Validation & Reach)
*   **Media Blitz:** Issue a comprehensive press release announcing 2-3 major keynote speakers (the "Wow Factor") to Tier 1 tech publications (TechCrunch, Wired, industry-specific journals).
*   **Strategic Partnerships (Co-Branding):** Secure one major industry player (e.g., a cloud provider, a major tech firm). Their participation lends immediate credibility and provides massive cross-promotion reach.

---

## 🤝 Phase 3: Attendee Engagement & Conversion Strategy

The goal is to move leads from *awareness* to *purchase* through added-value incentives.

| Stage | Activity | Objective | Measurement/KPI |
| :--- | :--- | :--- | :--- |
| **Pre-Event (6-8 Weeks Out)** | **Early Bird Tier Launch:** Limited to the first 100 bookings. Heavily marketed on LinkedIn. | Create FOMO (Fear of Missing Out) and secure initial baseline attendance. | *Ticket Sales Count / Early Bird Redemption Rate.* |
| **Mid-Event (3-5 Weeks Out)** | **The "Companion Ticket" Offer:** Incentivize group purchases (e.g., 3 tickets = discount + private executive lounge access). | Increase average ticket value and generate word-of-mouth selling. | *Average Tickets Purchased Per Group.* |
| **Last Call (1-2 Weeks Out)** | **The "Urgency Package":** Highlight the limited capacity of the networking space or exclusive workshops. Change messaging to "Almost Sold Out." | Final conversion push; capitalize on momentum. | *Last Minute Sales Velocity.* |
| **Virtual Touchpoint** | **Personalized Email Drip Campaign:** Sends different value propositions based on which web content the attendee clicked on (e.g., if they read about AI, send an AI-focused speaker announcement). | Keep the conference top-of-mind and nurture interest. | *Website Interaction Rate / Email Open Rate.* |

---

## 📈 Key Performance Indicators (KPIs) & Tracking

Success must be quantifiable. Our monitoring cycle will track progress against the 500-attendee goal weekly.

*   **Target:** 500+ Qualified Attendees
*   **Baseline Metric:** Email Lead Collection (Webinars/Downloads) $\rightarrow$ *Must exceed 1,500 leads.*
*   **Ad Campaign Efficiency:** Cost Per Lead (CPL) $\rightarrow$ *Must remain below industry average.*
*   **Conversion Rate:** (Initial Tickets Sold / Total Qualified Leads) $\rightarrow$ *Goal: 3.5% conversion rate.*

---

## ✅ Conclusion: Commitment to Excellence

This integrated strategy ensures that every marketing dollar spent drives not only *exposure* but genuine *engagement*. By building anticipation weeks in advance, providing unparalleled educational value, and maximizing the networking opportunity, we are confident that the initial goal of 500+ attendees will be met, setting a new standard for industry conferences.

**Recommendation:** Immediate approval to allocate resources for the LinkedIn advertising structure and the PR media outreach package.